In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import functions

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    CountVectorizer,
)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    ConfusionMatrixDisplay,
)

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

import re

In [15]:
df = pd.read_csv("../../data/clean_train.csv", sep=";",encoding="utf-8-sig", keep_default_na=False)

## Create Transcription Headers

In [16]:
df["transcription_headers"] = df["transcription"].apply(functions.extract_headers)
df["transcription_headers_model"] = df["transcription_headers"].apply(functions.headers_for_model)

# TF-IDF

### Train/Test Split

In [17]:
from sklearn.model_selection import train_test_split

train_indices, valid_indices = train_test_split(
    df.index,
    test_size=0.2,
    random_state=42,
    stratify=df["medical_specialty"],
)

y_train = df.loc[train_indices, "medical_specialty"]
y_valid = df.loc[valid_indices, "medical_specialty"]

### Classifiers

In [18]:
linear_svc = LinearSVC(C=0.5,class_weight="balanced",)
logistic_regression = LogisticRegression(C=1.0,class_weight="balanced",max_iter=2000,random_state=42,)
multinomail_nb = MultinomialNB()

### Linear SVC Classifier

In [26]:
input_headers = functions.build_input_variants(df, ["description", "sample_name", "transcription", "keywords", "transcription_headers_model"])
input = functions.build_input_variants(df, ["description", "sample_name", "transcription", "keywords"])


results = pd.DataFrame([
    functions.evaluate_input_extra(input_headers, "With Headers", linear_svc, "Linear SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input, "Without Headers", linear_svc, "Linear_SVC", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_headers, "With Headers", logistic_regression, "Logistic Regression", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input, "Without Headers", logistic_regression, "Logistic Regression", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input_headers, "With Headers", multinomail_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1],
    functions.evaluate_input_extra(input, "Without Headers", multinomail_nb, "Multinomial NB", train_indices, valid_indices, y_train, y_valid)[1]
])

results

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear SVC,With Headers,0.480300,0.521755,0.153846,Neurosurgery
1,Linear_SVC,Without Headers,0.478424,0.520257,0.153846,Neurosurgery
2,Logistic Regression,With Headers,0.529081,0.570128,0.363636,Neurosurgery
3,Logistic Regression,Without Headers,0.538462,0.575199,0.363636,Neurosurgery
4,Multinomial NB,With Headers,0.454034,0.175918,0.000000,Dermatology
5,Multinomial NB,Without Headers,0.448405,0.172935,0.000000,Dermatology


In [22]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_key = functions.build_input_variants(df, ["keywords"])

pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", linear_svc, "Linear_SVC", train_indices, valid_indices, y_train, y_valid)
pred2, result2 = functions.evaluate_input_extra(input_key, "key", linear_svc, "Linear_SVC", train_indices, valid_indices, y_train, y_valid)

labels = sorted(y_valid.unique())

functions.show_classification_report(
    y_valid,
    pred1,
    labels,
)
functions.show_classification_report(
    y_valid,
    pred2,
    labels,
)
results = pd.DataFrame([result1, result2])
results

                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.90      0.87      0.89        62
             Dermatology       0.80      0.80      0.80         5
        Gastroenterology       0.91      0.77      0.83        39
        General Medicine       0.67      0.88      0.76        42
               Neurology       0.72      0.78      0.75        37
            Neurosurgery       0.88      0.88      0.88        16
   Obstetrics-Gynecology       1.00      0.74      0.85        27
           Ophthalmology       0.88      0.93      0.90        15
              Orthopedic       0.88      0.88      0.88        60
   Psychiatry-Psychology       0.67      0.75      0.71         8
               Radiology       0.97      0.89      0.93        44
                 Surgery       0.94      0.95      0.95       178

                accuracy                           0.88       533
               macro avg       0.85      0.84      0.84       533
        

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Linear_SVC,key and Headers,0.879925,0.844047,0.705882,Psychiatry-Psychology
1,Linear_SVC,key,0.840525,0.822564,0.166667,Psychiatry-Psychology


In [24]:
input = functions.build_input_variants(df, ["keywords", "transcription_headers_model"])
input_key = functions.build_input_variants(df, ["keywords"])

pred1, result1 = functions.evaluate_input_extra(input, "key and Headers", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)
pred2, result2 = functions.evaluate_input_extra(input_key, "key", logistic_regression, "Log Regression", train_indices, valid_indices, y_train, y_valid)

labels = sorted(y_valid.unique())

functions.show_classification_report(
    y_valid,
    pred1,
    labels,
)
functions.show_classification_report(
    y_valid,
    pred2,
    labels,
)
results = pd.DataFrame([result1, result2])
results

                          precision    recall  f1-score   support

Cardiovascular-Pulmonary       0.90      0.87      0.89        62
             Dermatology       0.50      0.60      0.55         5
        Gastroenterology       0.79      0.77      0.78        39
        General Medicine       0.63      0.90      0.75        42
               Neurology       0.77      0.73      0.75        37
            Neurosurgery       0.50      0.88      0.64        16
   Obstetrics-Gynecology       0.77      0.74      0.75        27
           Ophthalmology       0.52      0.93      0.67        15
              Orthopedic       0.78      0.78      0.78        60
   Psychiatry-Psychology       0.70      0.88      0.78         8
               Radiology       0.90      0.86      0.88        44
                 Surgery       0.93      0.74      0.82       178

                accuracy                           0.79       533
               macro avg       0.72      0.81      0.75       533
        

,model,input,accuracy,macro_f1,lowest_f1,lowest_f1_label
0,Log Regression,key and Headers,0.793621,0.752410,0.545455,Dermatology
1,Log Regression,key,0.782364,0.764078,0.166667,Psychiatry-Psychology


In [ ]:
from collections import Counter

header_counts = Counter(
    header
    for headers in df["transcription_headers"]
    for header in headers
)

header_counts.most_common(20)

[('PROCEDURE', 848),
 ('ANESTHESIA', 839),
 ('PREOPERATIVE DIAGNOSIS', 769),
 ('POSTOPERATIVE DIAGNOSIS', 699),
 ('COMPLICATIONS', 523),
 ('IMPRESSION', 455),
 ('FINDINGS', 441),
 ('ESTIMATED BLOOD LOSS', 420),
 ('PROCEDURE PERFORMED', 415),
 ('POSTOPERATIVE DIAGNOSES', 340),
 ('PREOPERATIVE DIAGNOSES', 302),
 ('HISTORY OF PRESENT ILLNESS', 298),
 ('DESCRIPTION OF PROCEDURE', 276),
 ('HEENT', 271),
 ('EXAM', 268),
 ('INDICATIONS', 263),
 ('ALLERGIES', 253),
 ('MEDICATIONS', 248),
 ('PHYSICAL EXAMINATION', 246),
 ('PLAN', 242)]